# P2 — Vietnamese News Classification & Trend Analytics Platform

## Phase 2 — Vietnamese NLP Preprocessing & TF-IDF

**Dataset:** UVN-1  
**Dataset Version:** uvn1-v1.0.0  
**Random Seed:** 42

### Experiments
- E0 — Raw text baseline
- E1 — Underthesea tokenization
- E2 — Stopword removal
- E3 — TF-IDF grid search

> Test set remains LOCKED during Phase 2.

In [3]:
# ============================================================
# Section 0 — Setup & Config
# ============================================================

from pathlib import Path
import json
import time
import itertools
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

import underthesea
from underthesea import word_tokenize


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

STOPWORDS_PATH = PROJECT_ROOT / "data" / "stopwords_vi.txt"

TRAIN_PATH = DATA_PROCESSED / "train.csv"
VAL_PATH = DATA_PROCESSED / "validation.csv"
TEST_PATH = DATA_PROCESSED / "test.csv"

CATEGORIES_PATH = DATA_PROCESSED / "categories.json"

CACHE_PATH = DATA_INTERIM / "news_tokenized.csv"

METADATA_PATH = (
    DATA_PROCESSED / "phase2_preprocessing_metadata.json"
)


# ------------------------------------------------------------
# Versions
# ------------------------------------------------------------

DATASET_VERSION = "uvn1-v1.0.0"
STOPWORDS_VERSION = "vi-v1.0.0"
CACHE_FORMAT_VERSION = "cache-v2.0.0"
EXPECTED_UNDERTHESEA_VERSION = "9.5.0"

RANDOM_SEED = 42


# ------------------------------------------------------------
# Environment check
# ------------------------------------------------------------

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Underthesea:", underthesea.__version__)
print("Dataset version:", DATASET_VERSION)
print("Random seed:", RANDOM_SEED)

assert (
    underthesea.__version__
    == EXPECTED_UNDERTHESEA_VERSION
), (
    f"Expected Underthesea "
    f"{EXPECTED_UNDERTHESEA_VERSION}, "
    f"got {underthesea.__version__}"
)

print("\n✅ Section 0 — Setup & Config PASS")

PROJECT_ROOT: c:\Users\ADMIN\Desktop\Vietnamese News Classification
Underthesea: 9.5.0
Dataset version: uvn1-v1.0.0
Random seed: 42

✅ Section 0 — Setup & Config PASS


In [18]:
%pip install underthesea==9.5.0

  Using cached underthesea-9.5.0-py3-none-any.whl.metadata (10 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached underthesea_core-3.3.2-cp313-cp313-win_amd64.whl.metadata (5.8 kB)
  Using cached huggingface_hub-1.31.0-py3-none-any.whl.metadata (16 kB)
  Using cached filelock-3.32.6-py3-none-any.whl.metadata (2.0 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.5.1-cp313-cp313-win_amd64.whl.metadata (46 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached underthesea-9.5.0-py3-none-any.whl (7.3 MB)
Using cached underthesea_core-3.3.2-cp313-cp313-win_amd64.whl (1.2 MB)
Using cached huggingface_hub-1.31.0-py3-none-any.whl (798 kB)
Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl (4.0 MB)
Using cached filelock-3.32.6-py3-none-any.whl (100 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
Using cached re

## Section 1 — Load Data

Load only the training and validation sets.

- Train: 2,272 samples
- Validation: 487 samples
- Test: LOCKED — not loaded in Phase 2

In [5]:
# ============================================================
# Section 1 — Load Data
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH,
    encoding="utf-8-sig"
)

val_df = pd.read_csv(
    VAL_PATH,
    encoding="utf-8-sig"
)

# Basic checks
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nTrain classes:", train_df["label"].nunique())
print("Validation classes:", val_df["label"].nunique())

# Expected sizes
assert len(train_df) == 2272, (
    f"Expected 2272 train samples, got {len(train_df)}"
)

assert len(val_df) == 487, (
    f"Expected 487 validation samples, got {len(val_df)}"
)

# ID must not overlap
train_ids = set(train_df["id"].astype(str))
val_ids = set(val_df["id"].astype(str))

assert train_ids.isdisjoint(val_ids), (
    "Train and validation IDs overlap!"
)

# Required columns
required_columns = {"id", "title", "content", "label"}

assert required_columns.issubset(train_df.columns)
assert required_columns.issubset(val_df.columns)

print("\nTest: 🔒 LOCKED — not loaded in Phase 2")

print("\n✅ Section 1 — Load Data PASS")

Train shape: (2272, 7)
Validation shape: (487, 7)

Train columns:
['id', 'title', 'content', 'label', 'url', 'text', 'clean_text']

Validation columns:
['id', 'title', 'content', 'label', 'url', 'text', 'clean_text']

Train classes: 13
Validation classes: 13

Test: 🔒 LOCKED — not loaded in Phase 2

✅ Section 1 — Load Data PASS


## Section 2 — Build Text

Create the classification text from:

`title + content`

The original `title` and `content` columns remain unchanged.

In [6]:
# ============================================================
# Section 2 — Build Text
# ============================================================

def build_text(df):
    df = df.copy()

    df["title"] = df["title"].fillna("").astype(str)
    df["content"] = df["content"].fillna("").astype(str)

    df["text"] = (
        df["title"].str.strip()
        + " "
        + df["content"].str.strip()
    ).str.strip()

    return df


# Build text for train and validation
train_df = build_text(train_df)
val_df = build_text(val_df)


# Check
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nSample:")
print("TITLE:")
print(train_df.iloc[0]["title"][:200])

print("\nTEXT:")
print(train_df.iloc[0]["text"][:500])

# No empty text
assert train_df["text"].notna().all()
assert val_df["text"].notna().all()

assert (train_df["text"].str.len() > 0).all()
assert (val_df["text"].str.len() > 0).all()

print("\n✅ Section 2 — Build Text PASS")

Train shape: (2272, 7)
Validation shape: (487, 7)

Sample:
TITLE:
ABBANK được chấp thuận chào bán cổ phiếu để nâng vốn điều lệ

TEXT:
ABBANK được chấp thuận chào bán cổ phiếu để nâng vốn điều lệ Theo Giấy chứng nhận đăng ký chào bán thêm cổ phiếu ra công chúng số 563 ký ngày 31/12/2025 do UBCKNN cấp, ABBANK sẽ thực hiện tăng vốn điều lệ thêm hơn 3.105 tỷ đồng (tương đương tăng 30% vốn điều lệ hiện tại) thông qua hình thức chào bán cổ phiếu ra công chúng cho cổ đông hiện hữu theo phương thức thực hiện quyền. Cụ thể, với mỗi 100 cổ phiếu sở hữu tại ngày chốt danh sách, cổ đông được mua thêm 30 cổ phiếu mới. Số lượng chào bán là 

✅ Section 2 — Build Text PASS


## Section 3 — E0: Raw Text Baseline

Baseline:
- Raw `title + content`
- TF-IDF: `ngram_range=(1,1)`
- `min_df=2`
- `sublinear_tf=False`
- Logistic Regression sanity check
- TF-IDF fitted only on train
- Validation used only for evaluation

In [7]:
# ============================================================
# Section 3 — E0: Raw Text Baseline
# ============================================================

tfidf_e0 = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,
    sublinear_tf=False,
    lowercase=True
)

# Fit TF-IDF ONLY on training data
X_train_e0 = tfidf_e0.fit_transform(train_df["text"])

# Validation is transform only
X_val_e0 = tfidf_e0.transform(val_df["text"])

y_train = train_df["label"]
y_val = val_df["label"]

# Logistic Regression sanity check
lr_e0 = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED
)

lr_e0.fit(X_train_e0, y_train)

y_val_pred_e0 = lr_e0.predict(X_val_e0)

# Metrics
e0_macro_f1 = f1_score(
    y_val,
    y_val_pred_e0,
    average="macro"
)

e0_accuracy = accuracy_score(
    y_val,
    y_val_pred_e0
)

print("E0 — Raw Text Baseline")
print(f"Train matrix: {X_train_e0.shape}")
print(f"Validation matrix: {X_val_e0.shape}")
print(f"Features: {X_train_e0.shape[1]:,}")
print(f"Validation Macro-F1: {e0_macro_f1:.4f}")
print(f"Validation Accuracy: {e0_accuracy:.4f}")

print("\n✅ Section 3 — E0 PASS")

E0 — Raw Text Baseline
Train matrix: (2272, 8746)
Validation matrix: (487, 8746)
Features: 8,746
Validation Macro-F1: 0.6607
Validation Accuracy: 0.8275

✅ Section 3 — E0 PASS


## Section 4 — E1: Underthesea Tokenization

Apply Vietnamese word segmentation using Underthesea.

- Train + Validation only
- Cache tokenization results
- No stopword removal yet
- No test data
- Same TF-IDF baseline as E0
- Logistic Regression used only as a sanity check

In [8]:
# ============================================================
# Section 4 — E1: Underthesea Tokenization
# ============================================================

def cache_is_valid(cache_df, train_df, val_df):
    required = {
        "id",
        "tokenized_raw",
        "dataset_version",
        "underthesea_version",
        "cache_version",
        "split"
    }

    if cache_df.empty:
        return False

    if not required.issubset(cache_df.columns):
        return False

    # Check metadata
    meta_ok = (
        (cache_df["dataset_version"] == DATASET_VERSION).all()
        and
        (cache_df["underthesea_version"] == EXPECTED_UNDERTHESEA_VERSION).all()
        and
        (cache_df["cache_version"] == CACHE_FORMAT_VERSION).all()
    )

    if not meta_ok:
        return False

    # Expected IDs
    expected_train = set(
        train_df["id"].astype(str)
    )

    expected_val = set(
        val_df["id"].astype(str)
    )

    # Cached IDs
    cached_train = set(
        cache_df.loc[
            cache_df["split"] == "train",
            "id"
        ].astype(str)
    )

    cached_val = set(
        cache_df.loc[
            cache_df["split"] == "validation",
            "id"
        ].astype(str)
    )

    if expected_train != cached_train:
        return False

    if expected_val != cached_val:
        return False

    if cache_df["tokenized_raw"].isna().any():
        return False

    return True


# ------------------------------------------------------------
# Load existing cache if available
# ------------------------------------------------------------

cache_df = pd.DataFrame()

if CACHE_PATH.exists():
    cache_df = pd.read_csv(
        CACHE_PATH,
        encoding="utf-8-sig"
    )

    print("Existing cache found:")
    print(CACHE_PATH)

    if cache_is_valid(
        cache_df,
        train_df,
        val_df
    ):
        print("✅ Cache is VALID")
    else:
        print("⚠️ Cache is INVALID → rebuilding")
        cache_df = pd.DataFrame()
else:
    print("No existing cache found → building tokenization cache")


# ------------------------------------------------------------
# Build cache
# ------------------------------------------------------------

if cache_df.empty:

    cache_rows = []

    combined_df = pd.concat(
        [
            train_df[["id", "text"]].assign(
                split="train"
            ),
            val_df[["id", "text"]].assign(
                split="validation"
            )
        ],
        ignore_index=True
    )

    start_time = time.time()

    for idx, row in combined_df.iterrows():

        tokens = word_tokenize(
            str(row["text"])
        )

        cache_rows.append({
            "id": str(row["id"]),
            "tokenized_raw": json.dumps(
                tokens,
                ensure_ascii=False
            ),
            "dataset_version": DATASET_VERSION,
            "underthesea_version": EXPECTED_UNDERTHESEA_VERSION,
            "cache_version": CACHE_FORMAT_VERSION,
            "split": row["split"]
        })

        if (idx + 1) % 500 == 0:
            print(
                f"Tokenized: {idx + 1}/"
                f"{len(combined_df)}"
            )

    cache_df = pd.DataFrame(cache_rows)

    DATA_INTERIM.mkdir(
        parents=True,
        exist_ok=True
    )

    cache_df.to_csv(
        CACHE_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    elapsed = time.time() - start_time

    print(
        f"\nTokenization completed in "
        f"{elapsed:.2f} seconds"
    )

else:
    print("Using existing valid cache.")


print("\nCache shape:", cache_df.shape)
print("Cache columns:", cache_df.columns.tolist())


# ------------------------------------------------------------
# Convert cached token list → E1 text
# ------------------------------------------------------------

def build_tokenized_raw_text(tokenized_raw_json):

    tokens = json.loads(
        tokenized_raw_json
    )

    cleaned = []

    for tok in tokens:

        tok_lower = (
            str(tok)
            .lower()
            .strip()
        )

        if not tok_lower:
            continue

        if tok_lower.isspace():
            continue

        # Remove punctuation-only tokens
        if all(
            not ch.isalnum()
            for ch in tok_lower
        ):
            continue

        cleaned.append(tok_lower)

    return " ".join(cleaned)


cache_df["tokenized_raw_text"] = (
    cache_df["tokenized_raw"]
    .apply(build_tokenized_raw_text)
)


# ------------------------------------------------------------
# Map tokenized text back to train / validation
# ------------------------------------------------------------

token_map = cache_df.set_index(
    "id"
)["tokenized_raw_text"]


train_df["tokenized_raw_text"] = (
    train_df["id"]
    .astype(str)
    .map(token_map)
)

val_df["tokenized_raw_text"] = (
    val_df["id"]
    .astype(str)
    .map(token_map)
)


# Check
assert train_df[
    "tokenized_raw_text"
].notna().all()

assert val_df[
    "tokenized_raw_text"
].notna().all()


# ------------------------------------------------------------
# Show Underthesea example
# ------------------------------------------------------------

print("\nOriginal:")
print(train_df.iloc[0]["text"][:200])

print("\nUnderthesea tokenized:")
print(
    train_df.iloc[0]["tokenized_raw_text"][:300]
)


# ------------------------------------------------------------
# E1 TF-IDF
# ------------------------------------------------------------

tfidf_e1 = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,
    sublinear_tf=False,
    lowercase=False
)

X_train_e1 = tfidf_e1.fit_transform(
    train_df["tokenized_raw_text"]
)

X_val_e1 = tfidf_e1.transform(
    val_df["tokenized_raw_text"]
)


# ------------------------------------------------------------
# Logistic Regression sanity check
# ------------------------------------------------------------

lr_e1 = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED
)

lr_e1.fit(
    X_train_e1,
    y_train
)

y_val_pred_e1 = lr_e1.predict(
    X_val_e1
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

e1_macro_f1 = f1_score(
    y_val,
    y_val_pred_e1,
    average="macro"
)

e1_accuracy = accuracy_score(
    y_val,
    y_val_pred_e1
)


print("\nE1 — Underthesea Tokenization")

print(
    f"Train matrix: {X_train_e1.shape}"
)

print(
    f"Validation matrix: {X_val_e1.shape}"
)

print(
    f"Features: {X_train_e1.shape[1]:,}"
)

print(
    f"Validation Macro-F1: "
    f"{e1_macro_f1:.4f}"
)

print(
    f"Validation Accuracy: "
    f"{e1_accuracy:.4f}"
)

print("\nComparison:")
print(f"E0 Macro-F1: {e0_macro_f1:.4f}")
print(f"E1 Macro-F1: {e1_macro_f1:.4f}")
print(
    f"Change: "
    f"{e1_macro_f1 - e0_macro_f1:+.4f}"
)

print("\n✅ Section 4 — E1 PASS")

No existing cache found → building tokenization cache
Tokenized: 500/2759
Tokenized: 1000/2759
Tokenized: 1500/2759
Tokenized: 2000/2759
Tokenized: 2500/2759

Tokenization completed in 74.52 seconds

Cache shape: (2759, 6)
Cache columns: ['id', 'tokenized_raw', 'dataset_version', 'underthesea_version', 'cache_version', 'split']

Original:
ABBANK được chấp thuận chào bán cổ phiếu để nâng vốn điều lệ Theo Giấy chứng nhận đăng ký chào bán thêm cổ phiếu ra công chúng số 563 ký ngày 31/12/2025 do UBCKNN cấp, ABBANK sẽ thực hiện tăng vốn điề

Underthesea tokenized:
abbank được chấp thuận chào bán cổ phiếu để nâng vốn điều lệ theo giấy chứng nhận đăng ký chào bán thêm cổ phiếu ra công chúng số 563 ký ngày 31/12/2025 do ubcknn cấp abbank sẽ thực hiện tăng vốn điều lệ thêm hơn 3.105 tỷ đồng tương đương tăng 30 vốn điều lệ hiện tại thông qua hình thức chào bán cổ 

E1 — Underthesea Tokenization
Train matrix: (2272, 8648)
Validation matrix: (487, 8648)
Features: 8,648
Validation Macro-F1: 0.6607


## Section 5 — E2: Vietnamese Stopword Removal

Apply the curated Vietnamese stopword list after Underthesea tokenization.

- Underthesea tokenization is preserved
- Remove punctuation-only tokens
- Remove curated Vietnamese stopwords
- Keep domain-specific classification words
- Train + Validation only
- Same TF-IDF configuration as E0/E1
- Logistic Regression sanity check
- Test set remains LOCKED

In [9]:
# ============================================================
# Section 5 — E2: Vietnamese Stopword Removal
# ============================================================

assert STOPWORDS_PATH.exists(), (
    f"Stopword file not found: {STOPWORDS_PATH}"
)

# ------------------------------------------------------------
# Load stopwords
# ------------------------------------------------------------

with open(
    STOPWORDS_PATH,
    "r",
    encoding="utf-8-sig"
) as f:

    stopwords_vi = {
        line.strip().lower()
        for line in f
        if line.strip()
        and not line.lstrip().startswith("#")
    }

print("Stopwords loaded:", len(stopwords_vi))
print("Stopword file:", STOPWORDS_PATH)

assert len(stopwords_vi) > 0

# ------------------------------------------------------------
# Build E2 tokenized text
# ------------------------------------------------------------

def build_tokenized_text(
    tokenized_raw_json,
    stopwords
):

    tokens = json.loads(
        tokenized_raw_json
    )

    cleaned = []

    for tok in tokens:

        tok_lower = (
            str(tok)
            .lower()
            .strip()
        )

        # Empty token
        if not tok_lower:
            continue

        # Whitespace-only token
        if tok_lower.isspace():
            continue

        # Punctuation-only token
        if all(
            not ch.isalnum()
            for ch in tok_lower
        ):
            continue

        # Stopword removal
        if tok_lower in stopwords:
            continue

        cleaned.append(tok_lower)

    return " ".join(cleaned)


# Apply to cache
cache_df["tokenized_text"] = (
    cache_df["tokenized_raw"]
    .apply(
        lambda x: build_tokenized_text(
            x,
            stopwords_vi
        )
    )
)

# ------------------------------------------------------------
# Map E2 text back to train / validation
# ------------------------------------------------------------

token_map_e2 = cache_df.set_index(
    "id"
)["tokenized_text"]

train_df["tokenized_text"] = (
    train_df["id"]
    .astype(str)
    .map(token_map_e2)
)

val_df["tokenized_text"] = (
    val_df["id"]
    .astype(str)
    .map(token_map_e2)
)

assert train_df[
    "tokenized_text"
].notna().all()

assert val_df[
    "tokenized_text"
].notna().all()


# ------------------------------------------------------------
# Show example
# ------------------------------------------------------------

print("\nBefore stopword removal:")
print(
    train_df.iloc[0][
        "tokenized_raw_text"
    ][:400]
)

print("\nAfter stopword removal:")
print(
    train_df.iloc[0][
        "tokenized_text"
    ][:400]
)


# ------------------------------------------------------------
# E2 TF-IDF
# ------------------------------------------------------------

tfidf_e2 = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,
    sublinear_tf=False,
    lowercase=False
)

X_train_e2 = tfidf_e2.fit_transform(
    train_df["tokenized_text"]
)

X_val_e2 = tfidf_e2.transform(
    val_df["tokenized_text"]
)


# ------------------------------------------------------------
# Logistic Regression sanity check
# ------------------------------------------------------------

lr_e2 = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED
)

lr_e2.fit(
    X_train_e2,
    y_train
)

y_val_pred_e2 = lr_e2.predict(
    X_val_e2
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

e2_macro_f1 = f1_score(
    y_val,
    y_val_pred_e2,
    average="macro"
)

e2_accuracy = accuracy_score(
    y_val,
    y_val_pred_e2
)


print("\nE2 — Stopword Removal")

print(
    f"Train matrix: {X_train_e2.shape}"
)

print(
    f"Validation matrix: {X_val_e2.shape}"
)

print(
    f"Features: {X_train_e2.shape[1]:,}"
)

print(
    f"Validation Macro-F1: "
    f"{e2_macro_f1:.4f}"
)

print(
    f"Validation Accuracy: "
    f"{e2_accuracy:.4f}"
)


# ------------------------------------------------------------
# Compare E0 → E1 → E2
# ------------------------------------------------------------

print("\nComparison:")

print(
    f"E0 Macro-F1: "
    f"{e0_macro_f1:.4f}"
)

print(
    f"E1 Macro-F1: "
    f"{e1_macro_f1:.4f}"
)

print(
    f"E2 Macro-F1: "
    f"{e2_macro_f1:.4f}"
)

print(
    f"E2 vs E1: "
    f"{e2_macro_f1 - e1_macro_f1:+.4f}"
)

print(
    f"E2 vs E0: "
    f"{e2_macro_f1 - e0_macro_f1:+.4f}"
)

print("\n✅ Section 5 — E2 PASS")

Stopwords loaded: 145
Stopword file: c:\Users\ADMIN\Desktop\Vietnamese News Classification\data\stopwords_vi.txt

Before stopword removal:
abbank được chấp thuận chào bán cổ phiếu để nâng vốn điều lệ theo giấy chứng nhận đăng ký chào bán thêm cổ phiếu ra công chúng số 563 ký ngày 31/12/2025 do ubcknn cấp abbank sẽ thực hiện tăng vốn điều lệ thêm hơn 3.105 tỷ đồng tương đương tăng 30 vốn điều lệ hiện tại thông qua hình thức chào bán cổ phiếu ra công chúng cho cổ đông hiện hữu theo phương thức thực hiện quyền cụ thể với mỗi 100 cổ phiế

After stopword removal:
abbank chấp thuận chào bán cổ phiếu nâng vốn điều lệ giấy chứng nhận đăng ký chào bán thêm cổ phiếu công chúng số 563 ký ngày 31/12/2025 ubcknn cấp abbank thực hiện tăng vốn điều lệ thêm hơn 3.105 đồng tương đương tăng 30 vốn điều lệ thông qua hình thức chào bán cổ phiếu công chúng cổ đông hiện hữu phương thức thực hiện quyền cụ thể 100 cổ phiếu sở hữu ngày chốt danh sách cổ đông mua thêm 30 cổ ph

E2 — Stopword Removal
Train matri

## Section 6 — E3: TF-IDF Grid Search

Search 18 TF-IDF configurations:

- ngram_range: (1,1), (1,2), (1,3)
- min_df: 1, 2, 5
- sublinear_tf: False, True

Total configurations: 18

Each configuration:
- Fit TF-IDF on train only
- Transform validation
- Train Logistic Regression
- Evaluate Validation Macro-F1

The test set remains LOCKED.

In [10]:
# ============================================================
# Section 6 — E3: TF-IDF Grid Search
# ============================================================

# TF-IDF search space
ngram_ranges = [
    (1, 1),
    (1, 2),
    (1, 3)
]

min_dfs = [
    1,
    2,
    5
]

sublinear_options = [
    False,
    True
]

grid = list(
    itertools.product(
        ngram_ranges,
        min_dfs,
        sublinear_options
    )
)

print("Total TF-IDF configurations:", len(grid))

assert len(grid) == 18


# ------------------------------------------------------------
# Run grid search
# ------------------------------------------------------------

e3_results = []

start_time = time.time()

for config_id, (
    ngram_range,
    min_df,
    sublinear_tf
) in enumerate(grid, start=1):

    print(
        f"\n[{config_id:02d}/18] "
        f"ngram={ngram_range}, "
        f"min_df={min_df}, "
        f"sublinear_tf={sublinear_tf}"
    )

    # TF-IDF
    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        min_df=min_df,
        sublinear_tf=sublinear_tf,
        lowercase=False
    )

    X_train = vectorizer.fit_transform(
        train_df["tokenized_text"]
    )

    X_val = vectorizer.transform(
        val_df["tokenized_text"]
    )

    # Logistic Regression sanity check
    lr = LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED
    )

    lr.fit(
        X_train,
        y_train
    )

    y_pred = lr.predict(
        X_val
    )

    # Metrics
    macro_f1 = f1_score(
        y_val,
        y_pred,
        average="macro"
    )

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    result = {
        "config_id": config_id,
        "ngram_range": str(ngram_range),
        "min_df": min_df,
        "sublinear_tf": sublinear_tf,
        "features": X_train.shape[1],
        "macro_f1": macro_f1,
        "accuracy": accuracy
    }

    e3_results.append(result)

    print(
        f"    Features: {X_train.shape[1]:,}"
    )

    print(
        f"    Macro-F1: {macro_f1:.4f}"
    )

    print(
        f"    Accuracy: {accuracy:.4f}"
    )


elapsed = time.time() - start_time


# ------------------------------------------------------------
# Results DataFrame
# ------------------------------------------------------------

e3_results = pd.DataFrame(
    e3_results
)

e3_results = e3_results.sort_values(
    by="macro_f1",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 60)
print("E3 — TF-IDF GRID SEARCH RESULTS")
print("=" * 60)

print(
    f"Configurations tested: "
    f"{len(e3_results)}"
)

print(
    f"Total time: "
    f"{elapsed:.2f} seconds"
)


# ------------------------------------------------------------
# Top 5 configurations
# ------------------------------------------------------------

print("\nTop 5 configurations:")

display(
    e3_results.head(5)
)


# ------------------------------------------------------------
# Best configuration
# ------------------------------------------------------------

best_e3 = e3_results.iloc[0]

print("\nBest configuration:")
print(
    f"ngram_range : "
    f"{best_e3['ngram_range']}"
)

print(
    f"min_df      : "
    f"{best_e3['min_df']}"
)

print(
    f"sublinear_tf: "
    f"{best_e3['sublinear_tf']}"
)

print(
    f"Features     : "
    f"{int(best_e3['features']):,}"
)

print(
    f"Macro-F1     : "
    f"{best_e3['macro_f1']:.4f}"
)

print(
    f"Accuracy     : "
    f"{best_e3['accuracy']:.4f}"
)


# ------------------------------------------------------------
# Compare E0 → E1 → E2 → E3
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPERIMENT COMPARISON")
print("=" * 60)

print(
    f"E0 Raw             : "
    f"{e0_macro_f1:.4f}"
)

print(
    f"E1 Underthesea     : "
    f"{e1_macro_f1:.4f}"
)

print(
    f"E2 Stopwords       : "
    f"{e2_macro_f1:.4f}"
)

print(
    f"E3 Best TF-IDF     : "
    f"{best_e3['macro_f1']:.4f}"
)

print(
    f"\nE3 vs E0: "
    f"{best_e3['macro_f1'] - e0_macro_f1:+.4f}"
)

print(
    f"E3 vs E2: "
    f"{best_e3['macro_f1'] - e2_macro_f1:+.4f}"
)

print("\n✅ Section 6 — E3 PASS")

Total TF-IDF configurations: 18

[01/18] ngram=(1, 1), min_df=1, sublinear_tf=False
    Features: 20,984
    Macro-F1: 0.6518
    Accuracy: 0.8214

[02/18] ngram=(1, 1), min_df=1, sublinear_tf=True
    Features: 20,984
    Macro-F1: 0.5753
    Accuracy: 0.8255

[03/18] ngram=(1, 1), min_df=2, sublinear_tf=False
    Features: 8,637
    Macro-F1: 0.6626
    Accuracy: 0.8275

[04/18] ngram=(1, 1), min_df=2, sublinear_tf=True
    Features: 8,637
    Macro-F1: 0.5729
    Accuracy: 0.8234

[05/18] ngram=(1, 1), min_df=5, sublinear_tf=False
    Features: 4,547
    Macro-F1: 0.6673
    Accuracy: 0.8296

[06/18] ngram=(1, 1), min_df=5, sublinear_tf=True
    Features: 4,547
    Macro-F1: 0.5818
    Accuracy: 0.8296

[07/18] ngram=(1, 2), min_df=1, sublinear_tf=False
    Features: 386,423
    Macro-F1: 0.5266
    Accuracy: 0.7967

[08/18] ngram=(1, 2), min_df=1, sublinear_tf=True
    Features: 386,423
    Macro-F1: 0.4929
    Accuracy: 0.7803

[09/18] ngram=(1, 2), min_df=2, sublinear_tf=False
  

,config_id,ngram_range,min_df,sublinear_tf,features,macro_f1,accuracy
0,5,"(1, 1)",5,False,4547,0.667291,0.829569
1,3,"(1, 1)",2,False,8637,0.662586,0.827515
2,1,"(1, 1)",1,False,20984,0.651790,0.821355
3,11,"(1, 2)",5,False,37541,0.592792,0.823409
4,6,"(1, 1)",5,True,4547,0.581829,0.829569



Best configuration:
ngram_range : (1, 1)
min_df      : 5
sublinear_tf: False
Features     : 4,547
Macro-F1     : 0.6673
Accuracy     : 0.8296

EXPERIMENT COMPARISON
E0 Raw             : 0.6607
E1 Underthesea     : 0.6607
E2 Stopwords       : 0.6626
E3 Best TF-IDF     : 0.6673

E3 vs E0: +0.0066
E3 vs E2: +0.0047

✅ Section 6 — E3 PASS


In [11]:
# ============================================================
# E3 — Xem đầy đủ kết quả
# ============================================================

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Tổng số cấu hình:", len(e3_results))

display(
    e3_results[
        [
            "config_id",
            "ngram_range",
            "min_df",
            "sublinear_tf",
            "features",
            "macro_f1",
            "accuracy"
        ]
    ]
)

print("\n" + "=" * 60)
print("TOP 5 E3")
print("=" * 60)

display(e3_results.head(5))

best_e3 = e3_results.iloc[0]

print("\nBEST E3")
print("ngram_range :", best_e3["ngram_range"])
print("min_df      :", best_e3["min_df"])
print("sublinear_tf:", best_e3["sublinear_tf"])
print("Features     :", int(best_e3["features"]))
print("Macro-F1     :", f"{best_e3['macro_f1']:.4f}")
print("Accuracy     :", f"{best_e3['accuracy']:.4f}")

print("\n" + "=" * 60)
print("E0 → E1 → E2 → E3")
print("=" * 60)

print(f"E0 Raw         : {e0_macro_f1:.4f}")
print(f"E1 Underthesea : {e1_macro_f1:.4f}")
print(f"E2 Stopwords   : {e2_macro_f1:.4f}")
print(f"E3 Best        : {best_e3['macro_f1']:.4f}")

print(f"\nE3 vs E2: {best_e3['macro_f1'] - e2_macro_f1:+.4f}")
print(f"E3 vs E0: {best_e3['macro_f1'] - e0_macro_f1:+.4f}")

assert len(e3_results) == 18

print("\n✅ Section 6 — E3 PASS")

Tổng số cấu hình: 18


,config_id,ngram_range,min_df,sublinear_tf,features,macro_f1,accuracy
0,5,"(1, 1)",5,False,4547,0.667291,0.829569
1,3,"(1, 1)",2,False,8637,0.662586,0.827515
2,1,"(1, 1)",1,False,20984,0.651790,0.821355
3,11,"(1, 2)",5,False,37541,0.592792,0.823409
4,6,"(1, 1)",5,True,4547,0.581829,0.829569
5,12,"(1, 2)",5,True,37541,0.578059,0.823409
6,2,"(1, 1)",1,True,20984,0.575318,0.825462
7,4,"(1, 1)",2,True,8637,0.572924,0.823409
8,17,"(1, 3)",5,False,62663,0.556267,0.811088
9,18,"(1, 3)",5,True,62663,0.545382,0.809035



TOP 5 E3


,config_id,ngram_range,min_df,sublinear_tf,features,macro_f1,accuracy
0,5,"(1, 1)",5,False,4547,0.667291,0.829569
1,3,"(1, 1)",2,False,8637,0.662586,0.827515
2,1,"(1, 1)",1,False,20984,0.651790,0.821355
3,11,"(1, 2)",5,False,37541,0.592792,0.823409
4,6,"(1, 1)",5,True,4547,0.581829,0.829569



BEST E3
ngram_range : (1, 1)
min_df      : 5
sublinear_tf: False
Features     : 4547
Macro-F1     : 0.6673
Accuracy     : 0.8296

E0 → E1 → E2 → E3
E0 Raw         : 0.6607
E1 Underthesea : 0.6607
E2 Stopwords   : 0.6626
E3 Best        : 0.6673

E3 vs E2: +0.0047
E3 vs E0: +0.0066

✅ Section 6 — E3 PASS


## Section 7 — Save Tokenized Dataset

Save the final Phase 2 tokenized datasets.

- Train: 2,272 samples
- Validation: 487 samples
- Test: LOCKED — not loaded in Phase 2

Saved fields:
- id
- title
- content
- label
- url
- text
- tokenized_text

The original `title`, `content`, and `text` are preserved.

In [12]:
# ============================================================
# Section 7 — Save Tokenized Dataset
# ============================================================

OUTPUT_COLUMNS = [
    "id",
    "title",
    "content",
    "label",
    "url",
    "text",
    "tokenized_text"
]

train_tokenized = train_df[OUTPUT_COLUMNS].copy()
val_tokenized = val_df[OUTPUT_COLUMNS].copy()

train_output_path = DATA_PROCESSED / "train_tokenized.csv"
val_output_path = DATA_PROCESSED / "validation_tokenized.csv"

train_tokenized.to_csv(
    train_output_path,
    index=False,
    encoding="utf-8-sig"
)

val_tokenized.to_csv(
    val_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Train tokenized:")
print("  Shape:", train_tokenized.shape)
print("  Path :", train_output_path)

print("\nValidation tokenized:")
print("  Shape:", val_tokenized.shape)
print("  Path :", val_output_path)

print("\nColumns:")
print(train_tokenized.columns.tolist())

# Validation
assert train_tokenized.shape[0] == 2272
assert val_tokenized.shape[0] == 487

assert train_tokenized["tokenized_text"].notna().all()
assert val_tokenized["tokenized_text"].notna().all()

assert train_tokenized["id"].is_unique
assert val_tokenized["id"].is_unique

assert set(train_tokenized["id"]) == set(train_df["id"])
assert set(val_tokenized["id"]) == set(val_df["id"])

assert train_output_path.exists()
assert val_output_path.exists()

print("\n✅ Section 7 — Save Tokenized Dataset PASS")

Train tokenized:
  Shape: (2272, 7)
  Path : c:\Users\ADMIN\Desktop\Vietnamese News Classification\data\processed\train_tokenized.csv

Validation tokenized:
  Shape: (487, 7)
  Path : c:\Users\ADMIN\Desktop\Vietnamese News Classification\data\processed\validation_tokenized.csv

Columns:
['id', 'title', 'content', 'label', 'url', 'text', 'tokenized_text']

✅ Section 7 — Save Tokenized Dataset PASS


## Section 8 — Export Phase 2 Metadata

Save the reproducibility metadata for Phase 2.

- Dataset version
- Train/validation sizes
- Number of classes
- Underthesea version
- Stopword version and count
- Cache format version
- E0 → E3 validation Macro-F1
- Best TF-IDF configuration
- Random seed
- Test set remains LOCKED

In [13]:
# ============================================================
# Section 8 — Export Phase 2 Metadata
# ============================================================

best_e3 = e3_results.iloc[0]

metadata = {
    "dataset_version": DATASET_VERSION,
    "train_samples": len(train_df),
    "validation_samples": len(val_df),

    # Test is intentionally not loaded in Phase 2
    "test_samples": None,
    "test_status": "LOCKED — not loaded in Phase 2",

    "num_classes": int(train_df["label"].nunique()),

    "underthesea_version": EXPECTED_UNDERTHESEA_VERSION,

    "stopwords_version": STOPWORDS_VERSION,
    "stopwords_count": int(len(stopwords_vi)),

    "cache_format_version": CACHE_FORMAT_VERSION,

    "punctuation_removal": True,

    "experiments": [
        "E0",
        "E1",
        "E2",
        "E3"
    ],

    "e3_grid_size": 18,

    "val_macro_f1_e0_to_e3": {
        "E0": float(e0_macro_f1),
        "E1": float(e1_macro_f1),
        "E2": float(e2_macro_f1),
        "E3_best": float(best_e3["macro_f1"])
    },

    "val_accuracy_e0_to_e3": {
        "E0": float(e0_accuracy),
        "E1": float(e1_accuracy),
        "E2": float(e2_accuracy),
        "E3_best": float(best_e3["accuracy"])
    },

    "best_tfidf_config": {
        "ngram_range": list(best_e3["ngram_range"]),
        "min_df": int(best_e3["min_df"]),
        "sublinear_tf": bool(best_e3["sublinear_tf"]),
        "features": int(best_e3["features"]),
        "macro_f1": float(best_e3["macro_f1"]),
        "accuracy": float(best_e3["accuracy"])
    },

    "random_seed": RANDOM_SEED
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Metadata saved to:")
print(METADATA_PATH)

print("\nSummary:")
print(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2
    )
)

assert METADATA_PATH.exists()

print("\n✅ Section 8 — Export Metadata PASS")

Metadata saved to:
c:\Users\ADMIN\Desktop\Vietnamese News Classification\data\processed\phase2_preprocessing_metadata.json

Summary:
{
  "dataset_version": "uvn1-v1.0.0",
  "train_samples": 2272,
  "validation_samples": 487,
  "test_samples": null,
  "test_status": "LOCKED — not loaded in Phase 2",
  "num_classes": 13,
  "underthesea_version": "9.5.0",
  "stopwords_version": "vi-v1.0.0",
  "stopwords_count": 145,
  "cache_format_version": "cache-v2.0.0",
  "punctuation_removal": true,
  "experiments": [
    "E0",
    "E1",
    "E2",
    "E3"
  ],
  "e3_grid_size": 18,
  "val_macro_f1_e0_to_e3": {
    "E0": 0.6607031652345456,
    "E1": 0.6607031652345456,
    "E2": 0.6625861820619048,
    "E3_best": 0.6672909910716954
  },
  "val_accuracy_e0_to_e3": {
    "E0": 0.8275154004106776,
    "E1": 0.8275154004106776,
    "E2": 0.8275154004106776,
    "E3_best": 0.8295687885010267
  },
  "best_tfidf_config": {
    "ngram_range": [
      "(",
      "1",
      ",",
      " ",
      "1",
      ")